In [3]:
pip install opencv-python

   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.2/38.8 MB 3.5 MB/s eta 0:00:12
    --------------------------------------- 0.6/38.8 MB 5.4 MB/s eta 0:00:08
    --------------------------------------- 0.9/38.8 MB 6.0 MB/s eta 0:00:07
   - -------------------------------------- 1.1/38.8 MB 6.2 MB/s eta 0:00:07
   - -------------------------------------- 1.1/38.8 MB 6.2 MB/s eta 0:00:07
   - -------------------------------------- 1.2/38.8 MB 4.0 MB/s eta 0:00:10
   - -------------------------------------- 1.5/38.8 MB 4.4 MB/s eta 0:00:09
   - -------------------------------------- 1.9/38.8 MB 4.8 MB/s eta 0:00:08
   -- ------------------------------------- 2.2/38.8 MB 5.1 MB/s eta 0:00:08
   -- ------------------------------------- 2.6/38.8 MB 5.3 MB/s eta 0:00:07
   --- ------------------------------------ 3.0/38.8 MB 5.6 MB/s eta 0:00:07
   --- ------

In [7]:
pip install torchvision 

  Using cached torchvision-0.20.1-cp311-cp311-win_amd64.whl.metadata (6.2 kB)
  Using cached torch-2.5.1-cp311-cp311-win_amd64.whl.metadata (28 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.6 MB 393.8 kB/s eta 0:00:04
   -------- ------------------------------- 0.3/1.6 MB 2.2 MB/s eta 0:00:01
   ----------------- ---------------------- 0.7/1.6 MB 3.9 MB/s eta 0:00:01
   -------------------------- ------------- 1.0/1.6 MB 4.7 MB/s eta 0:00:01
   ----------------------------------- ---- 1.4/1.6 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 5.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/203.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/203.1 

In [8]:
import itertools
import re

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

import cv2
from PIL import Image
from skimage.feature import hog

from sklearn import preprocessing
from sklearn import svm
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import SubsetRandomSampler, DataLoader
from torchvision import transforms, models

In [10]:
path = "C:/Users/PC/Downloads/all_data_info.csv/all_data_info.csv"

df = pd.read_csv(path)
df.head()

,artist,date,genre,pixelsx,pixelsy,size_bytes,source,style,title,artist_group,in_train,new_filename
0,Barnett Newman,1955.0,abstract,15530.0,6911.0,9201912.0,wikiart,Color Field Painting,Uriel,train_only,True,102257.jpg
1,Barnett Newman,1950.0,abstract,14559.0,6866.0,8867532.0,wikiart,Color Field Painting,Vir Heroicus Sublimis,train_only,True,75232.jpg
2,kiri nichol,2013.0,NaN,9003.0,9004.0,1756681.0,NaN,Neoplasticism,NaN,test_only,False,32145.jpg
3,kiri nichol,2013.0,NaN,9003.0,9004.0,1942046.0,NaN,Neoplasticism,NaN,test_only,False,20304.jpg
4,kiri nichol,2013.0,NaN,9003.0,9004.0,1526212.0,NaN,Neoplasticism,NaN,test_only,False,836.jpg


In [11]:
print(f"The full dataset contains a total of {len(df['artist'].unique())} different artists and {len(df['genre'].unique())} unique painting genres.\n")
ash = 5
print(f"The {ash} artists with the most paintings available in the dataset are:")
df['artist'].value_counts().head(ash)

The full dataset contains a total of 2319 different artists and 43 unique painting genres.

The 5 artists with the most paintings available in the dataset are:


artist
Ivan Aivazovsky          500
John Singer Sargent      500
Pierre-Auguste Renoir    500
Marc Chagall             500
Pablo Picasso            500
Name: count, dtype: int64

In [12]:
painter_dict = {'Chagall':'','Dali':'','Picasso':'','Delacroix':'','Rembrandt':'','Gogh':'', 'Goya':'',
               'Modigliani':'','Pissarro':'','Renoir':'', 'Degas':''}

paintings_dict = painter_dict.copy()

# Find the correspondence between our names and the names in the dataframe
# Also count the number of paintings
for artist in painter_dict:
    for painter in df['artist']:
        if artist in painter:
            painter_dict[artist] = painter
            paintings = df[df['artist'] == painter].shape[0]
            paintings_dict[artist] = paintings
            break
for artist in painter_dict:
    print(f'The artist named {painter_dict[artist]} has a total of {paintings_dict[artist]} paintings in this dataset.')

sample_size = min(paintings_dict.values())
min_a = list(paintings_dict.keys())[list(paintings_dict.values()).index(sample_size)]
print(f'\nThe artist with the smallest number of paintings is {min_a} with {sample_size} paintings.')

The artist named Marc Chagall has a total of 500 paintings in this dataset.
The artist named Salvador Dali has a total of 485 paintings in this dataset.
The artist named Pablo Picasso has a total of 500 paintings in this dataset.
The artist named Eugene Delacroix has a total of 221 paintings in this dataset.
The artist named Rembrandt has a total of 500 paintings in this dataset.
The artist named Vincent van Gogh has a total of 494 paintings in this dataset.
The artist named Francisco Goya has a total of 386 paintings in this dataset.
The artist named Amedeo Modigliani has a total of 342 paintings in this dataset.
The artist named Camille Pissarro has a total of 499 paintings in this dataset.
The artist named Pierre-Auguste Renoir has a total of 500 paintings in this dataset.
The artist named Edgar Degas has a total of 495 paintings in this dataset.

The artist with the smallest number of paintings is Delacroix with 221 paintings.


In [13]:
paintings = df['artist'].value_counts().head(20)
artists = paintings.index.tolist()
sample_size = min(paintings)

In [14]:
active_df = pd.DataFrame({}) # Reduce the large dataframe into the one containing only relevant data

#for artist in artists: # <- use this in the boring case
for artist in painter_dict.values(): # <- use this in the non boring case
    tr_df = df[(df['artist']==artist)].sort_values(by=['in_train','size_bytes'], ascending=[False, True])
    active_df = pd.concat([active_df,tr_df.iloc[:sample_size]])

In [15]:
artists = list(painter_dict.values())

# Label Encoder to transform artist names into integers from 0 to 19
LabEnc = preprocessing.LabelEncoder()
LabEnc.fit(artists)

LabelEncoder()

In [17]:
matplotlib.rc_file_defaults()

def image_transformer_nn(image, apply_norm=True, crop_img=True, new_dim=224):
    """
    Args:
        resize_num (int):
            Dimension (pixels) to resize image
        apply_norm (bool):
            Choose whether to apply the normalization or not
        crop_img (bool):
            Choose whether to resize the image into the new_dim size, or crop
            a square from its center, sized new_dim x new_dim
    """
    if crop_img:
        cropper = transforms.CenterCrop(new_dim)
        image = cropper(image)
    # Using transforms.Compose() is another option to perform these sequentially, but
    # let's keep it like this until we find the "final" transformations sequence
    tensoring = transforms.ToTensor()
    image = tensoring(image) # shape is now (channels, height, width), see next line
    channels, height, width = image.shape
    
    # This check was added because some images are automatically loaded as grayscale
    if image.shape[0] < 3:
        image = image.expand(3, -1, -1)
    # This check is for images like 18807 that have extra channels with zero information
    if image.shape[0] > 3:
        image = image[0:3,:,:]
    
    # This is the imagenet normalizer, maybe define our own?
    if apply_norm:
        normalizer = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        image = normalizer(image)
    
    if not crop_img:
        if width < height:
            # Convolutionsare invariant to rotations, so we choose to pad everything
            # "down". This means that for landscape images (width > height) no rotation
            # needs to be performed we just pad "down". For vertical images (width < height),
            # in order to perform the "down" padding we have to rotate them first.
            image = image.transpose(1,2)
        channels, height, width = image.shape
        res_percent = float(new_dim/width) # done to keep aspect ratio, width is max dim
        height = round(height*res_percent)
        resizer = transforms.Resize((height,new_dim))
        image = resizer(image)
        # Now that the image is resized by keeping aspect ratio, we pad "down"
        padder = transforms.Pad([0,0,0,int(new_dim-height)])
        image = padder(image)
        
    return image

# Example:
archive = zipfile.ZipFile(file_path+'train.zip', 'r')
img_path = 'train/'
imgdata = archive.open(img_path+'69382.jpg')
image = Image.open(imgdata)

print("This is the original image:\n")
plt.imshow(image)
plt.show()

print("This is the cropped part of the transformed image:\n")
image2 = image_transformer_nn(image, apply_norm=False, crop_img=True, new_dim=224)
plt.imshow(image2.numpy().transpose(1,2,0))
plt.show()

print("This is the transformed, resized image:\n")
image3a = image_transformer_nn(image, apply_norm=False, crop_img=False, new_dim=224)
plt.imshow(image3a.numpy().transpose(1,2,0))
plt.show()

NameError: name 'zipfile' is not defined